# CIFAR-100 — Multi-Exit Architecture Training

This notebook trains three multi-exit architectures on CIFAR-100 from scratch.
All models share the same training recipe and exit head design.
No gating mechanism is applied here — this notebook is purely about training.
Gating (energy vs entropy comparison) happens in Notebook 2.

**Models trained in order of increasing training time:**

| Order | Model | Params | Est. time/epoch |
|-------|-------|--------|----------------|
| 1 | Multi-exit ResNet-18 | ~11M | ~50s |
| 2 | Multi-exit ResNet-50 | ~25M | ~190s |
| 3 | Multi-exit WideResNet-28-10 | ~36M | ~280s |

**What all three models share:**
- Same CIFAR stem fix (3×3 conv, no maxpool)
- Same exit head design at each exit point
- Same curriculum-weighted multi-exit loss
- Same training recipe: SGD + OneCycleLR + CutMix + label smoothing
- 4 exit points placed at natural layer boundaries

**Checkpoints saved:** `best_me_resnet18.pth`, `best_me_resnet50.pth`, `best_me_wrn2810.pth`

## 0 — Imports and device

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
    print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 1 — Data loading

In [ ]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True,  download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(trainset):,} samples | {len(trainloader)} batches')
print(f'Test : {len(testset):,} samples | {len(testloader)} batches')

## 2 — Shared components

### Exit head
Every exit point across all three architectures uses this identical head.
Global average pool collapses spatial dimensions, then a two-layer MLP
with BN and dropout produces class logits.
The hidden dimension (512) is fixed regardless of the input channel count.

### CutMix
Applied randomly to 50% of batches. Pastes a rectangular crop from one
image into another and mixes labels proportionally to patch area.

### Curriculum loss weights
Loss weights shift across three training phases so earlier exits receive
stronger supervision early in training when the final exit hasn't converged yet.

| Phase | Epochs (of total) | w1 | w2 | w3 | w4 |
|-------|-------------------|----|----|----|----|----|
| Early | 0–30% | 0.40 | 0.30 | 0.20 | 0.10 |
| Mid   | 30–65% | 0.25 | 0.30 | 0.25 | 0.20 |
| Late  | 65–100% | 0.10 | 0.20 | 0.30 | 0.40 |

In [ ]:
# Exit head (shared by all architectures) 
class ExitHead(nn.Module):
    def __init__(self, in_channels, hidden=512, num_classes=100, dropout=0.3):
        super().__init__()
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(in_channels, hidden)
        self.bn      = nn.BatchNorm1d(hidden)
        self.drop    = nn.Dropout(dropout)
        self.fc2     = nn.Linear(hidden, num_classes)

    def forward(self, x):
        x = self.flatten(self.pool(x))
        return self.fc2(self.drop(F.relu(self.bn(self.fc1(x)))))


# CutMix
def cutmix_data(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = images.shape
    idx = torch.randperm(B).to(images.device)
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_ratio), int(H * cut_ratio)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w//2, 0, W); x2 = np.clip(cx + cut_w//2, 0, W)
    y1 = np.clip(cy - cut_h//2, 0, H); y2 = np.clip(cy + cut_h//2, 0, H)
    mixed = images.clone()
    mixed[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return mixed, labels, labels[idx], lam


# Curriculum loss weights
def get_loss_weights(epoch, total_epochs):
    frac = epoch / total_epochs
    if   frac < 0.30: return [0.40, 0.30, 0.20, 0.10]
    elif frac < 0.65: return [0.25, 0.30, 0.25, 0.20]
    else:             return [0.10, 0.20, 0.30, 0.40]


# Universal training loop
# All three architectures use this identical loop.
# The model.forward() must accept no threshold argument and return
# a list of 4 logit tensors — this is enforced by the architecture definitions below.

def train_one_epoch(model, optimizer, scheduler, criterion, epoch, total_epochs):
    model.train()
    total_loss = 0.0
    correct = 0; total = 0
    weights = get_loss_weights(epoch, total_epochs)

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        use_cutmix = np.random.rand() > 0.5
        if use_cutmix:
            images, la, lb, lam = cutmix_data(images, labels)

        optimizer.zero_grad()
        exit_logits = model(images)   # always returns list of 4 tensors

        loss = 0.0
        for w, logits in zip(weights, exit_logits):
            if use_cutmix:
                loss += w * (lam * criterion(logits, la) + (1-lam) * criterion(logits, lb))
            else:
                loss += w * criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        _, pred = exit_logits[-1].max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()

    return total_loss / len(trainloader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0; total = 0
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)[-1]   # final exit only for training eval
        _, pred = logits.max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()
    return 100.0 * correct / total


def run_training(model, save_path, total_epochs, patience=25):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=0.1, epochs=total_epochs,
        steps_per_epoch=len(trainloader),
        pct_start=0.1, anneal_strategy='cos',
        div_factor=10, final_div_factor=1e4
    )
    best_acc = 0.0; patience_counter = 0

    for epoch in range(1, total_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, optimizer, scheduler, criterion, epoch, total_epochs)
        test_acc = evaluate(model)
        elapsed  = time.time() - t0
        w = get_loss_weights(epoch, total_epochs)
        print(f'Epoch {epoch:3d}/{total_epochs} | loss {train_loss:.4f} | '
              f'train {train_acc:.2f}% | test {test_acc:.2f}% | w={w} | {elapsed:.0f}s')
        if test_acc > best_acc:
            best_acc = test_acc; patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f'  --> saved (best {best_acc:.2f}%)')
        else:
            patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stop at epoch {epoch}. Best: {best_acc:.2f}%')
            break

    print(f'\nFinal best test accuracy: {best_acc:.2f}%')
    return best_acc


print('Shared components defined.')

## 3 — Architecture 1: Multi-exit ResNet-18

**Estimated training time: ~80 min on T4**

ResNet-18 has 4 layer groups. Exit heads are attached after each one.
The CIFAR stem replaces the 7×7/stride-2 conv + maxpool with a 3×3/stride-1 conv,
preventing the aggressive downsampling that destroys spatial detail on 32×32 images.

```
stem (3×3, 64)  →  no maxpool
  layer1: 2 BasicBlocks, 64-d,   32×32  →  Exit 1  (~18% MACs)
  layer2: 2 BasicBlocks, 128-d,  16×16  →  Exit 2  (~32% MACs)
  layer3: 2 BasicBlocks, 256-d,   8×8   →  Exit 3  (~57% MACs)
  layer4: 2 BasicBlocks, 512-d,   4×4   →  Exit 4  (100% MACs)
```

In [ ]:
class MultiExitResNet18(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        base = models.resnet18(weights=None)
        # CIFAR stem: 3×3 conv, stride 1, no maxpool
        self.stem   = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), base.bn1, base.relu)
        self.layer1 = base.layer1   # [B,  64, 32, 32]
        self.layer2 = base.layer2   # [B, 128, 16, 16]
        self.layer3 = base.layer3   # [B, 256,  8,  8]
        self.layer4 = base.layer4   # [B, 512,  4,  4]
        self.exit1  = ExitHead( 64, 256, num_classes)
        self.exit2  = ExitHead(128, 256, num_classes)
        self.exit3  = ExitHead(256, 512, num_classes)
        self.exit4  = ExitHead(512, 512, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):   nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x  = self.stem(x)
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [self.exit1(f1), self.exit2(f2), self.exit3(f3), self.exit4(f4)]


me_rn18 = MultiExitResNet18().to(device)
p = sum(x.numel() for x in me_rn18.parameters())
print(f'MultiExitResNet18 — {p/1e6:.2f}M parameters')
# sanity check
with torch.no_grad():
    out = me_rn18(torch.randn(2,3,32,32).to(device))
print('Output shapes:', [o.shape for o in out])

In [ ]:
print('=' * 60)
print('TRAINING: Multi-Exit ResNet-18')
print('=' * 60)
best_rn18 = run_training(me_rn18, 'best_me_resnet18.pth', total_epochs=100, patience=20)

## 4 — Architecture 2: Multi-exit ResNet-50

**Estimated training time: ~360 min on T4**

ResNet-50 uses bottleneck blocks (1×1 → 3×3 → 1×1) instead of BasicBlocks.
Output channels after each group are 4× larger than ResNet-18 due to the
expansion factor in bottleneck design.

```
stem (3×3, 64)  →  no maxpool
  layer1: 3 Bottlenecks, 256-d,  32×32  →  Exit 1  (~20% MACs)
  layer2: 4 Bottlenecks, 512-d,  16×16  →  Exit 2  (~38% MACs)
  layer3: 6 Bottlenecks, 1024-d,  8×8   →  Exit 3  (~65% MACs)
  layer4: 3 Bottlenecks, 2048-d,  4×4   →  Exit 4  (100% MACs)
```

In [ ]:
class MultiExitResNet50(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        base = models.resnet50(weights=None)
        self.stem   = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), base.bn1, base.relu)
        self.layer1 = base.layer1   # [B,  256, 32, 32]
        self.layer2 = base.layer2   # [B,  512, 16, 16]
        self.layer3 = base.layer3   # [B, 1024,  8,  8]
        self.layer4 = base.layer4   # [B, 2048,  4,  4]
        self.exit1  = ExitHead( 256, 512, num_classes)
        self.exit2  = ExitHead( 512, 512, num_classes)
        self.exit3  = ExitHead(1024, 512, num_classes)
        self.exit4  = ExitHead(2048, 512, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):   nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x  = self.stem(x)
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [self.exit1(f1), self.exit2(f2), self.exit3(f3), self.exit4(f4)]


me_rn50 = MultiExitResNet50().to(device)
p = sum(x.numel() for x in me_rn50.parameters())
print(f'MultiExitResNet50 — {p/1e6:.2f}M parameters')
with torch.no_grad():
    out = me_rn50(torch.randn(2,3,32,32).to(device))
print('Output shapes:', [o.shape for o in out])

In [ ]:
print('=' * 60)
print('TRAINING: Multi-Exit ResNet-50')
print('=' * 60)
best_rn50 = run_training(me_rn50, 'best_me_resnet50.pth', total_epochs=200, patience=30)

## 5 — Training summary

In [ ]:
print('=' * 56)
print('TRAINING COMPLETE — FINAL EXIT ACCURACY SUMMARY')
print('=' * 56)
print(f'  Multi-Exit ResNet-18     : {best_rn18:.2f}%')
print(f'  Multi-Exit ResNet-50     : {best_rn50:.2f}%')
print(f'  Multi-Exit WRN-28-10     : {best_wrn:.2f}%')
print('=' * 56)
print()
print('Checkpoints saved:')
print('  best_me_resnet18.pth')
print('  best_me_resnet50.pth')
print('  best_me_wrn2810.pth')
print()
print('Next: load these checkpoints in Notebook 2 for')
print('energy vs entropy gating comparison.')